<a href="https://colab.research.google.com/github/obeabi/ProjectPortfolio/blob/master/TQQQ_TLT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade yfinance
!pip install  --upgrade pandas_ta
!pip install ta pandas_ta

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 59.9 MB/s eta 0:00:00
  Attempting uninstall: curl_cffi
    Found existing installation: curl_cffi 0.14.0
    Uninstalling curl_cffi-0.14.0:
      Successfully uninstalled curl_cffi-0.14.0
  Attempting uninstall: yfinance
    Found existing installation: yfinance 0.2.66
    Uninstalling yfinance-0.2.66:
      Successfully uninstalled yfinance-0.2.66
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.3/240.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 109.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Preparing metadata (setup.py) ... done
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=cc8bb674714cce97de153e1409a79a25f4361614d32841534a78521336144d51
  Stored in directory: /root/.cache/pip/wheels/5c/a1/5f/c6b85a7d9452057be4ce68a8e45d77ba34234a6d46581777c6
Successfully built ta


In [1]:
import yfinance as yf
print(yf.__version__)
import pandas as pd
import numpy as np
import seaborn as sns
from datetime import datetime
import time
import ta
import matplotlib.pyplot as plt
pd.set_option('display.max_colwidth', None)
print("Libraries Installed!")

1.2.1
Libraries Installed!


In [ ]:
# Download data
tickers = ["TQQQ", "TLT"]
data = yf.download(tickers, start="2024-01-01", interval="1d",auto_adjust=True)["Close"]

# Compute returns
returns = data.pct_change()

# Lookback window (3 months ~ 63 trading days)
window = 63

# Inspect dataframe
data.head()

In [ ]:

def compute_relative_strength(prices, method="raw"):
    roc = prices.pct_change(window) # 3-month return

    if method == "raw":
        rs = roc
    elif method == "vol_adj":
        vol = returns.rolling(window).std() * np.sqrt(window)
        rs = roc / vol
    else:
        raise ValueError("Method must be 'raw' or 'vol_adj'")

    return rs


    # Compute signals
rs_raw = compute_relative_strength(data, method="raw")
rs_vol = compute_relative_strength(data, method="vol_adj")



In [ ]:
# Resample monthly (take last available signal each month)
rs_raw_m = rs_raw.resample("ME").last()

#print("Relative strengths are :",rs_raw_m )
rs_raw_m.tail(10)

In [ ]:
# Resample monthly (take last available signal each month)
rs_vol_m = rs_vol.resample("ME").last()
#print("Volatility normalized Relative strengths are :",rs_vol_m )
rs_vol_m.tail(10)

In [ ]:
def generate_signals(rs):
    # Create a Series filled with 'CASH' for rows that are all NaN
    signals = pd.Series("CASH", index=rs.index)

    # Identify rows that are not all NaN
    valid_rows = rs.notna().any(axis=1)

    # For valid rows, pick the strongest asset
    signals.loc[valid_rows] = rs[valid_rows].idxmax(axis=1)

    # Apply volatility filter: go to CASH if max < 0
    signals.loc[rs.max(axis=1) < 0] = "CASH"

    return signals

signals_raw = generate_signals(rs_raw_m)
signals_vol = generate_signals(rs_vol_m)

# Strategy returns
monthly_returns = returns.resample("ME").apply(lambda x: (1+x).prod() - 1)

monthly_returns


In [ ]:
signals_vol.index[:-1]

signals_vol.loc['2025-11-30']

In [ ]:
def backtest(signals, cap_tqqq=False):
    strat_rets = []
    for date in signals.index[:-1]:
        asset = signals.loc[date]
        next_ret = monthly_returns.loc[date]

        if asset == "CASH":
            strat_rets.append(0)
        elif asset == "TQQQ" and cap_tqqq:
            strat_rets.append(0.7 * next_ret["TQQQ"]) # 50% cap
        elif asset == "TQQQ":
            # Instead of 50% TQQQ + 50% cash
            strat_rets.append(0.5 * next_ret["TQQQ"] + 0.5 * next_ret["QQQ"])
        else:
            strat_rets.append(next_ret[asset])

    return pd.Series(strat_rets, index=signals.index[:-1])

# Backtests
bt_raw = backtest(signals_raw, cap_tqqq=True)
bt_vol = backtest(signals_vol, cap_tqqq=True)

# Equity curves
equity_raw = (1 + bt_raw).cumprod()
equity_vol = (1 + bt_vol).cumprod()

In [ ]:

plt.figure(figsize=(12,6))

# Raw signals: solid line
plt.plot(equity_raw, label="Raw Signals", linewidth=2)

# Volatility signals: dotted line with square markers
plt.plot(
    equity_vol,
    label="Volatility Signals",
    linestyle=":",   # dotted line
    marker="s",     # square markers
    markersize=6,
    linewidth=1.5
)

plt.title("Equity Curves of Strategy", fontsize=16)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Equity (Cumulative Returns)", fontsize=12)
plt.legend()
plt.grid(True)
plt.show()



In [ ]:


def evaluate_strategy(strategy_returns, periods_per_year=12, risk_free=0.0):
    """
    Evaluate a strategy's performance metrics.

    Parameters:
    - strategy_returns: pd.Series of returns (monthly or daily)
    - periods_per_year: 12 for monthly, 252 for daily
    - risk_free: annual risk-free rate (default 0)

    Returns:
    - pd.Series with key metrics
    """

    # Remove NaNs
    returns = strategy_returns.dropna()

    # Cumulative return
    cumulative = (1 + returns).prod() - 1

    # Annualized return
    ann_return = (1 + cumulative) ** (periods_per_year / len(returns)) - 1

    # Annualized volatility
    ann_vol = returns.std() * np.sqrt(periods_per_year)

    # Sharpe ratio
    sharpe = (ann_return - risk_free) / ann_vol if ann_vol != 0 else np.nan

    # Equity curve & drawdowns
    equity = (1 + returns).cumprod()
    drawdown = equity / equity.cummax() - 1
    max_dd = drawdown.min()

    # Calmar ratio
    calmar = ann_return / abs(max_dd) if max_dd != 0 else np.nan

    # Sortino ratio
    downside_std = returns[returns < 0].std() * np.sqrt(periods_per_year)
    sortino = (ann_return - risk_free) / downside_std if downside_std != 0 else np.nan

    # Win rate / hit ratio
    win_rate = (returns > 0).mean()

    metrics = {
        "Cumulative Return": cumulative,
        "Annualized Return": ann_return,
        "Annualized Volatility": ann_vol,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_dd,
        "Calmar Ratio": calmar,
        "Sortino Ratio": sortino,
        "Hit Ratio": win_rate
    }

    return pd.Series(metrics)

# Example usage
metrics_raw = evaluate_strategy(bt_raw)
metrics_vol = evaluate_strategy(bt_vol)

# Combine into one table
metrics_comparison = pd.DataFrame({
    "Raw RS": metrics_raw,
    "Volatility-Weighted RS": metrics_vol
})

metrics_comparison


In [ ]:
# Step 1: Fetch data
tickers = ["TQQQ", "TLT"]
monthly_prices = yf.download(tickers, start="2019-01-01", interval="1mo", auto_adjust=True)["Close"]
# Drop missing values
monthly_prices = monthly_prices.dropna()


# ====================== RELATIVE STRENGTH ======================
signals = pd.DataFrame(index=monthly_prices.index)
signals["RS_1M"] = monthly_prices["TQQQ"].pct_change(1) - monthly_prices["TLT"].pct_change(1)
signals["RS_3M"] = monthly_prices["TQQQ"].pct_change(3) - monthly_prices["TLT"].pct_change(3)
signals["RS_6M"] = monthly_prices["TQQQ"].pct_change(6) - monthly_prices["TLT"].pct_change(6)

# Smoothed versions (very useful)
signals["RS_1M_SMA3"] = signals["RS_1M"].rolling(3).mean()
signals["RS_3M_SMA3"] = signals["RS_3M"].rolling(3).mean()
signals["RS_6M_SMA3"] = signals["RS_6M"].rolling(3).mean()

# ====================== REGIME + DECISION LOGIC ======================
def get_regime_and_decision(df):
    signals = df.copy()

    # Regime based on 3-month smoothed RS (primary trend filter)
    signals["Regime"] = pd.NA
    signals.loc[signals["RS_3M_SMA3"] > 0, "Regime"] = "Risk-On (favor TQQQ)"
    signals.loc[signals["RS_3M_SMA3"] < 0, "Regime"] = "Risk-Off (favor TLT)"

    # Final Decision
    signals["Decision"] = "WAIT (no alignment)"
    signals["Regime"]   = "Neutral"
    signals["Strength"] = "Weak"
    signals["Position"] = 0   # 1 = TQQQ, -1 = TLT, 0 = Cash

    # Strong TQQQ signal
    signals.loc[(signals["RS_3M_SMA3"] > 0) & (signals["RS_1M_SMA3"] > 0) & (signals["RS_6M_SMA3"] > 0), ["Decision", "Position"]] = ["ENTER TQQQ ✅", 1]

    # Strong TLT signal
    signals.loc[(signals["RS_3M_SMA3"] < 0) & (signals["RS_1M_SMA3"] < 0) & (signals["RS_6M_SMA3"] < 0), ["Decision", "Position"]] = ["ENTER TLT ✅", -1]

    # Optional: Add confidence level
    signals["Strength"] = "Weak"
    signals.loc[(signals["RS_3M_SMA3"] > 0) & (signals["RS_1M_SMA3"] > 0.015), "Strength"] = "Strong"
    signals.loc[(signals["RS_3M_SMA3"] < 0) & (signals["RS_1M_SMA3"] < -0.015), "Strength"] = "Strong"

    # ====================== BACKTEST ======================
    # Calculate monthly returns for both assets
    monthly_returns = monthly_prices.pct_change()

    # Strategy returns: Use previous month's position for current month's return (no lookahead)
    signals["Strategy_Return"] = signals["Position"].shift(1) * monthly_returns["TQQQ"] + \
                             (signals["Position"].shift(1) == -1) * monthly_returns["TLT"] * (-1)   # when Position=-1, we are long TLT
    # Fill NaN with 0 (no position = 0% return)
    signals["Strategy_Return"] = signals["Strategy_Return"].fillna(0)

    # Cumulative equity curves
    signals["TQQQ_BuyHold"] = (1 + monthly_returns["TQQQ"]).cumprod()
    signals["Strategy_Equity"] = (1 + signals["Strategy_Return"]).cumprod()

    # ====================== PERFORMANCE SUMMARY ======================
    print("=== PERFORMANCE SUMMARY (since 2019) ===")
    print(f"Final TQQQ Buy & Hold     : {signals['TQQQ_BuyHold'].iloc[-1]:.2f}x")
    print(f"Final Strategy Equity     : {signals['Strategy_Equity'].iloc[-1]:.2f}x")
    print(f"Strategy vs TQQQ Outperformance: {((signals['Strategy_Equity'].iloc[-1] / signals['TQQQ_BuyHold'].iloc[-1]) - 1)*100:.1f}%")

    return signals

signals = get_regime_and_decision(signals)

# Drop rows with NaN in key columns
signals = signals.dropna(subset=["RS_1M", "RS_3M"])


# ====================== OUTPUT ======================
print("Last 12 Months - TQQQ vs TLT Rotation Signals")
display_cols = ["RS_1M_SMA3", "RS_3M_SMA3", "Regime", "Decision","Strength", "Position", "Strategy_Return"]


signals[display_cols].tail(12).round(4)

In [ ]:
signals['Decision'].value_counts().plot(kind='bar')
plt.show()

In [ ]:
# ====================== PLOT EQUITY CURVES ======================
plt.figure(figsize=(12, 6))
plt.plot(signals.index, signals["TQQQ_BuyHold"], label="TQQQ Buy & Hold", linewidth=2)
plt.plot(signals.index, signals["Strategy_Equity"], label="RS Rotation Strategy (TQQQ/TLT)", linewidth=2, color='orange')
plt.title("Equity Curve: TQQQ vs Relative Strength Rotation Strategy")
plt.legend()
plt.grid(True)
plt.ylabel("Growth of $1")
plt.show()

In [ ]:
# Step 1: Fetch data
tickers = ["TQQQ", "BIL"]
monthly_prices = yf.download(tickers, start="2019-01-01", interval="1mo", auto_adjust=True)["Close"]
# Drop missing values
monthly_prices = monthly_prices.dropna()


# ====================== RELATIVE STRENGTH ======================
signals = pd.DataFrame(index=monthly_prices.index)
signals["RS_1M"] = monthly_prices["TQQQ"].pct_change(1) - monthly_prices["BIL"].pct_change(1)
signals["RS_3M"] = monthly_prices["TQQQ"].pct_change(3) - monthly_prices["BIL"].pct_change(3)
signals["RS_6M"] = monthly_prices["TQQQ"].pct_change(6) - monthly_prices["BIL"].pct_change(6)

# Smoothed versions (very useful)
signals["RS_1M_SMA3"] = signals["RS_1M"].rolling(3).mean()
signals["RS_3M_SMA3"] = signals["RS_3M"].rolling(3).mean()
signals["RS_6M_SMA3"] = signals["RS_6M"].rolling(3).mean()

# ====================== REGIME + DECISION LOGIC ======================
def get_regime_and_decision(df):
    signals = df.copy()

    # Regime based on 3-month smoothed RS (primary trend filter)
    signals["Regime"] = pd.NA
    signals["Decision"] = "ENTER BIL ✅"
    #signals["Regime"]   = "Neutral"
    signals["Strength"] = "Weak"
    signals["Position"] = 0   # 1 = TQQQ, 0 = BIL
    signals.loc[signals["RS_3M_SMA3"] > 0, "Regime"] = "Risk-On (favor TQQQ)"
    signals.loc[signals["RS_3M_SMA3"] < 0, "Regime"] = "Risk-Off (favor BIL)"

    # Strong TQQQ signal
    signals.loc[(signals["RS_3M_SMA3"] > 0.008) & (signals["RS_1M_SMA3"] > 0.01) & (signals["RS_6M_SMA3"] > -0.02), ["Decision", "Position"]] = ["ENTER TQQQ ✅", 1]


    # Optional: Add confidence level
    signals.loc[(signals["RS_3M_SMA3"] > 0.008) & (signals["RS_1M_SMA3"] > 0.01) & (signals["RS_6M_SMA3"] > -0.02), "Strength"] = "Strong"
    signals.loc[(signals["RS_3M_SMA3"] < 0) & (signals["RS_1M_SMA3"] < 0), "Strength"] = "Strong"

    # ====================== BACKTEST ======================
    # Calculate monthly returns for both assets
    monthly_returns = monthly_prices.pct_change()

    # Strategy returns: Use previous month's position for current month's return (no lookahead)
    signals["Strategy_Return"] = signals["Position"].shift(1) * monthly_returns["TQQQ"] + \
                             (signals["Position"].shift(1) == 0) * monthly_returns["BIL"]
    # Fill NaN with 0 (no position = 0% return)
    signals["Strategy_Return"] = signals["Strategy_Return"].fillna(0)
    signals["Vol_Scaler"] = 1.0
    signals.loc[signals["RS_3M_SMA3"].rolling(6).std() > 0.08, "Vol_Scaler"] = 0.5   # cut size in half when volatile
    signals["Strategy_Return"] = signals["Strategy_Return"] * signals["Vol_Scaler"].shift(1)

    # Cumulative equity curves

    signals["TQQQ_BuyHold"] = (1 + monthly_returns["TQQQ"]).cumprod()
    signals["Strategy_Equity"] = (1 + signals["Strategy_Return"]).cumprod()
    #signals["Strategy_Equity"] = (1 + signals["Strategy_Return"]).cumprod()
    #signals["Drawdown"] = signals["Equity"] / signals["Equity"].cummax() - 1
    # If drawdown > 15%, go to BIL immediately
    #signals["Position"] = np.where(signals["Drawdown"].shift(1) < -0.15, 0, signals["Position"])
    signals["BIL_Equity"] = (1 + monthly_returns["BIL"]).cumprod()

    # ====================== PERFORMANCE SUMMARY ======================
    print("=== PERFORMANCE SUMMARY (since 2019) ===")
    print("=== TQQQ vs BIL Rotation Strategy ===")
    print(f"Final TQQQ Buy & Hold     : {signals['TQQQ_BuyHold'].iloc[-1]:.2f}x")
    print(f"Final Strategy (TQQQ/BIL) : {signals['Strategy_Equity'].iloc[-1]:.2f}x")
    print(f"Final BIL                 : {signals['BIL_Equity'].iloc[-1]:.2f}x\n")
    print(f"Strategy vs TQQQ Outperformance: {((signals['Strategy_Equity'].iloc[-1] / signals['TQQQ_BuyHold'].iloc[-1]) - 1)*100:.1f}%")

    return signals

signals = get_regime_and_decision(signals)

# Drop rows with NaN in key columns
signals = signals.dropna(subset=["RS_1M", "RS_3M"])


# ====================== OUTPUT ======================
print("Last 12 Months - TQQQ vs BIL Rotation Signals")
display_cols = ["RS_1M_SMA3", "RS_3M_SMA3","RS_6M_SMA3","Vol_Scaler", "Regime", "Decision","Strength", "Position", "Strategy_Return"]


signals[display_cols].tail(10).round(4)

In [ ]:
signals['Decision'].value_counts().plot(kind='bar')
plt.show()

In [ ]:
# ====================== PLOT EQUITY CURVES ======================
plt.figure(figsize=(12, 6))
plt.plot(signals.index, signals["TQQQ_BuyHold"], label="TQQQ Buy & Hold", linewidth=2)
plt.plot(signals.index, signals["Strategy_Equity"], label="TQQQ / BIL Rotation Strategy", linewidth=2, color='orange')
plt.plot(signals.index, signals["BIL_Equity"], label="BIL", linewidth=1.5, color='gray', linestyle='--')
plt.title("Equity Curve: TQQQ vs TQQQ/BIL Rotation Strategy")
plt.legend()
plt.grid(True)
plt.ylabel("Growth of $1")
plt.show()

In [ ]:


def calculate_tqqq_bil_metrics(signals: pd.DataFrame, risk_free_rate: float = 0.0) -> pd.DataFrame:
    """
    Calculates comprehensive performance metrics for the TQQQ/BIL rotation strategy.

    Parameters:
        signals (pd.DataFrame): Your signals DataFrame that must contain 'Strategy_Return' column
        risk_free_rate (float): Annual risk-free rate (default 0.0). Use 0.04 for ~4% T-bill rate.

    Returns:
        pd.DataFrame with all metrics (nice formatted table)
    """

    if 'Strategy_Return' not in signals.columns:
        raise ValueError("DataFrame must contain 'Strategy_Return' column")

    returns = signals['Strategy_Return'].dropna()
    equity = (1 + returns).cumprod()

    # Basic calculations
    total_return = equity.iloc[-1] - 1
    num_months = len(returns)
    num_years = num_months / 12

    # Annualized Return (CAGR)
    cagr = (equity.iloc[-1]) ** (1 / num_years) - 1

    # Annualized Volatility
    ann_vol = returns.std() * np.sqrt(12)

    # Sharpe Ratio
    excess_returns = returns - (risk_free_rate / 12)
    sharpe = excess_returns.mean() / returns.std() * np.sqrt(12) if returns.std() != 0 else 0

    # Sortino Ratio
    downside_returns = returns[returns < 0]
    downside_std = downside_returns.std() if len(downside_returns) > 0 else 0
    sortino = excess_returns.mean() / downside_std * np.sqrt(12) if downside_std != 0 else 0

    # Max Drawdown
    running_max = equity.cummax()
    drawdown = equity / running_max - 1
    max_dd = drawdown.min()

    # Calmar Ratio
    calmar = cagr / abs(max_dd) if max_dd != 0 else np.inf

    # Win Rate / Hit Ratio
    win_rate = (returns > 0).mean() * 100

    # Average Profit, Median Profit
    positive_returns = returns[returns > 0]
    avg_profit = positive_returns.mean() if len(positive_returns) > 0 else 0
    median_profit = positive_returns.median() if len(positive_returns) > 0 else 0

    # Average Loss
    negative_returns = returns[returns < 0]
    avg_loss = negative_returns.mean() if len(negative_returns) > 0 else 0

    # Average Profit to Average Loss
    profit_to_loss = abs(avg_profit / avg_loss) if avg_loss != 0 else np.inf

    # Risk-Reward Ratio (RR)
    rr_ratio = abs(avg_profit / avg_loss) if avg_loss != 0 else np.inf

    # Profit Factor
    profit_factor = positive_returns.sum() / abs(negative_returns.sum()) if len(negative_returns) > 0 else np.inf

    # Create results table
    metrics = {
        "Metric": [
            "Cumulative Return",
            "CAGR (Annualized Return)",
            "Annualized Volatility",
            "Sharpe Ratio",
            "Sortino Ratio",
            "Max Drawdown",
            "Calmar Ratio",
            "Win Rate / Hit Ratio",
            "Average Profit (per winning month)",
            "Median Profit",
            "Avg Profit / Avg Loss",
            "Risk-Reward Ratio (RR)",
            "Profit Factor"
        ],
        "Value": [
            f"{total_return:.1%}",
            f"{cagr:.1%}",
            f"{ann_vol:.1%}",
            f"{sharpe:.2f}",
            f"{sortino:.2f}",
            f"{max_dd:.1%}",
            f"{calmar:.2f}",
            f"{win_rate:.1f}%",
            f"{avg_profit:.2%}",
            f"{median_profit:.2%}",
            f"{profit_to_loss:.2f}",
            f"{rr_ratio:.2f}",
            f"{profit_factor:.2f}"
        ]
    }

    result_df = pd.DataFrame(metrics)
    result_df.set_index("Metric", inplace=True)

    print("=== TQQQ / BIL Rotation Strategy Performance Metrics ===")
    print(f"Period: {signals.index[0].strftime('%Y-%m')} to {signals.index[-1].strftime('%Y-%m')}")
    print(f"Number of months: {num_months}\n")

    return result_df

In [ ]:
metrics = calculate_tqqq_bil_metrics(signals, risk_free_rate=0.04)  # Use 0.04 if you want realistic T-bill rate
print(metrics)

In [9]:
import pandas as pd
import yfinance as yf
import numpy as np

# ====================== DATA DOWNLOAD ======================
print("Downloading data...")
tickers = ["QQQ", "BIL", "QQQ"]
data = yf.download(tickers, start="2019-01-01", interval="1d", auto_adjust=True)["Close"].dropna()

# ====================== DAILY 50-DAY SMA SLOPE ON QQQ ======================
qqq_sma50 = data["QQQ"].rolling(50).mean()

def rolling_slope(series, window=10):
    def calc(y):
        if len(y) < 2:
            return np.nan
        x = np.arange(len(y))
        return np.polyfit(x, y, 1)[0]
    return series.rolling(window).apply(calc, raw=False)

qqq_50d_slope = rolling_slope(qqq_sma50, window=10)

# ====================== MONTHLY SIGNALS ======================
monthly_prices = data.resample('ME').last()

signals = pd.DataFrame(index=monthly_prices.index)

signals["RS_1M"] = monthly_prices["QQQ"].pct_change(1) - monthly_prices["BIL"].pct_change(1)
signals["RS_3M"] = monthly_prices["QQQ"].pct_change(3) - monthly_prices["BIL"].pct_change(3)
signals["RS_6M"] = monthly_prices["QQQ"].pct_change(6) - monthly_prices["BIL"].pct_change(6)

signals["RS_1M_SMA3"] = signals["RS_1M"].rolling(3).mean()
signals["RS_3M_SMA3"] = signals["RS_3M"].rolling(3).mean()
signals["RS_6M_SMA3"] = signals["RS_6M"].rolling(3).mean()

# ====================== TUNABLE THRESHOLDS ======================
RS_3M_THRESHOLD = 0.005
RS_1M_THRESHOLD = 0.008
RS_6M_THRESHOLD = -0.01     # Can be 0.0 if you want stricter

# ====================== ENTRY LOGIC (Monthly) ======================
signals["Position_Monthly"] = 0
signals["Reason"] = "HOLD BIL"

enter_condition = (
    (signals["RS_3M_SMA3"] > RS_3M_THRESHOLD) &
    (signals["RS_1M_SMA3"] > RS_1M_THRESHOLD) &
    (signals["RS_6M_SMA3"] > RS_6M_THRESHOLD)
)

signals.loc[enter_condition, ["Position_Monthly", "Reason"]] = [1, "ENTER TQQQ (RS 1M+3M+6M OK)"]

# ====================== DAILY EXECUTION WITH IMMEDIATE SLOPE EXIT ======================
daily_position = signals["Position_Monthly"].reindex(data.index, method='ffill').fillna(0)

# Immediate exit if 50-day SMA slope turns negative
exit_condition = (daily_position == 1) & (qqq_50d_slope < 0.0)

final_daily_position = daily_position.where(~exit_condition, 0)

# ====================== DAILY RETURNS ======================
daily_returns = (data["QQQ"].pct_change() * final_daily_position.shift(1) +
                 data["BIL"].pct_change() * (final_daily_position.shift(1) == 0))
daily_returns = daily_returns.fillna(0)

# ====================== EQUITY ======================
strategy_equity = (1 + daily_returns).cumprod()
tqqq_equity = (1 + data["QQQ"].pct_change()).cumprod()

# ====================== FULL METRICS ======================
def calculate_metrics(returns, equity, name):
    num_years = len(returns) / 252.0
    cagr = equity.iloc[-1] ** (1/num_years) - 1
    vol = returns.std() * np.sqrt(252)
    max_dd = (equity / equity.cummax() - 1).min()

    rf = 0.04 / 252
    sharpe = (returns.mean() - rf) / returns.std() * np.sqrt(252) if returns.std() != 0 else 0
    downside = returns[returns < 0]
    sortino = (returns.mean() - rf) / downside.std() * np.sqrt(252) if len(downside) > 0 else 0
    calmar = cagr / abs(max_dd) if max_dd != 0 else np.inf

    win_rate = (returns > 0).mean() * 100

    metrics = {
        "Cumulative Return": f"{equity.iloc[-1]-1:.1%}",
        "CAGR": f"{cagr:.1%}",
        "Volatility": f"{vol:.1%}",
        "Sharpe": f"{sharpe:.2f}",
        "Sortino": f"{sortino:.2f}",
        "Max Drawdown": f"{max_dd:.1%}",
        "Calmar": f"{calmar:.2f}",
        "Win Rate": f"{win_rate:.1f}%"
    }
    return pd.Series(metrics, name=name)

# Run and display
strat_metrics = calculate_metrics(daily_returns, strategy_equity, "QQQ/BIL + 6M RS + Daily Slope Exit")
tqqq_metrics  = calculate_metrics(data["QQQ"].pct_change(), tqqq_equity, "TQQQ Buy & Hold")

print("\n" + "="*65)
print("FINAL PERFORMANCE COMPARISON")
print("="*65)
print(pd.concat([strat_metrics, tqqq_metrics], axis=1))

# Recent signals
print("\nLast 10 Monthly Signals:")
print(signals[["RS_6M_SMA3", "RS_3M_SMA3", "RS_1M_SMA3", "Reason", "Position_Monthly"]].tail(10).round(4))

[*********************100%***********************]  2 of 2 completed



FINAL PERFORMANCE COMPARISON
                  QQQ/BIL + 6M RS + Daily Slope Exit TQQQ Buy & Hold
Cumulative Return                              78.5%          312.2%
CAGR                                            8.3%           21.6%
Volatility                                     13.1%           24.0%
Sharpe                                          0.37            0.77
Sortino                                         0.34            1.00
Max Drawdown                                  -14.7%          -35.1%
Calmar                                          0.57            0.61
Win Rate                                       58.8%           56.5%

Last 10 Monthly Signals:
            RS_6M_SMA3  RS_3M_SMA3  RS_1M_SMA3                       Reason  \
Date                                                                          
2025-07-31      0.0419      0.1197      0.0564  ENTER TQQQ (RS 1M+3M+6M OK)   
2025-08-31      0.0765      0.1453      0.0290  ENTER TQQQ (RS 1M+3M+6M OK)   
2025-09

In [ ]:

def get_correlation(tickers, period="3mo", interval="1d"):
    """
    Finds correlations.

    Parameters:
    -----------
    tickers : list
        All candidate tickers.
    period : str
        Data period for yfinance (default "3mo").
    interval : str
        Data interval (default "1d").

    Returns:
    --------
    corr_matrix : DataFrame
        Correlation matrix of daily returns.
    """
    # Step 1: Get prices
    data = yf.download(tickers, period=period, interval=interval,auto_adjust=True)["Close"]
    data = data.ffill()

    # Step 2: Convert to daily returns
    returns = data.pct_change().dropna()

    # Step 3: Correlation matrix
    corr_matrix = returns.corr()


     # Step 4: Plot heatmap
    plt.figure(figsize=(8,6))
    sns.heatmap(
        corr_matrix,
        annot=True,
        cmap="coolwarm",
        center=0,
        vmin=-1, vmax=1,
        linewidths=0.5
    )
    plt.title("Correlation Heatmap of Daily Returns based on 3 Months Performance", fontsize=14)
    plt.show()

    return corr_matrix


# Example usage
#tickers = ['PLTR', 'NVDA', 'MSFT', 'HOOD', 'GM','AAPL','GE','JPM', 'QQQ']  # Replace with your list of tickers
tickers = ['GS', 'MS', 'MSFT', 'JPM', 'GM', 'DJIA']  # Replace with your list of tickers
ranked_picks = tickers

corr_matrix = get_correlation(tickers)


corr_matrix

import investpy

data = investpy.get_stock_historical_data(
    stock='Fidelity Bank',
    country='Nigeria',
    from_date='01/01/2023',
    to_date='03/10/2025'
)
print(data.tail())
